# Custom Iterators and the Iteration Protocol

**Session:** 2026-10-30 — OOP with Python  
**Module:** 01 — Python Fundamentals & Algorithmic Thinking

Iteration is a foundational mechanism in Python. From simple `for` loops to PyTorch `DataLoader` streams, the execution flow is governed by the **Iteration Protocol**. In this notebook, we dissect how Python implements iteration under the hood, explore the contract between **Iterables** and **Iterators**, desugar loop syntax, and build custom stateful iterator classes.

## Learning Outcomes

- Distinguish clearly between an **Iterable** (`__iter__`) and an **Iterator** (`__next__`).
- Desugar `for` loops into explicit `iter()` calls, `while` loops, and `StopIteration` exception handling.
- Verify iteration mechanics across Python's built-in collections.
- Implement stateful, bounds-checked custom iterator classes from scratch.
- Design reusable, multi-pass iterables versus one-shot consumable iterators.
- Contrast iterator classes with generator functions (`yield`).

## 1. The Iteration Protocol Contract

Python defines the iteration protocol through two special dunder methods:

1. **`__iter__()` (Iterable):**
   - An object is an *Iterable* if it implements `__iter__()` returning an *Iterator*, or `__getitem__()` with sequential integer indices.
   - Calling `iter(obj)` invokes `obj.__iter__()`.

2. **`__next__()` (Iterator):**
   - An object is an *Iterator* if it implements `__next__()` and `__iter__()` (which returns `self`).
   - Each call to `next(it)` returns the next item in the sequence.
   - When no more elements remain, it must raise the `StopIteration` exception.

In [ ]:
# Inspecting built-in collections
sample_list = [10, 20, 30]

# List is an Iterable, not an Iterator
assert hasattr(sample_list, "__iter__")
assert not hasattr(sample_list, "__next__")

# Getting the iterator from the iterable
list_iterator = iter(sample_list)
assert hasattr(list_iterator, "__next__")
assert hasattr(list_iterator, "__iter__")
assert iter(list_iterator) is list_iterator  # Iterator returns self

# Fetch elements sequentially
print("First call:", next(list_iterator))
print("Second call:", next(list_iterator))
print("Third call:", next(list_iterator))

# Subsequent call raises StopIteration
try:
    next(list_iterator)
except StopIteration:
    print("Caught StopIteration: sequence exhausted.")

## 2. Desugaring the Python `for` Loop

What actually happens when Python executes `for item in collection:`? The interpreter transforms the syntax into an explicit `iter()` initialization and a `while True` loop guarded by `try...except StopIteration`:

In [ ]:
def simulate_for_loop(iterable) -> list:
    """Emulate Python's for loop using the iteration protocol explicitly."""
    collected = []
    iterator = iter(iterable)  # Step 1: obtain iterator

    while True:
        try:
            element = next(iterator)  # Step 2: fetch next item
        except StopIteration:
            break  # Step 3: terminate cleanly on exhaust
        collected.append(element)

    return collected


test_data = {"learning_rate": 0.001, "batch_size": 64, "optimizer": "Adam"}
result = simulate_for_loop(test_data)
print(f"Emulated loop output: {result}")
assert result == list(test_data.keys())

## 3. Implementing a Custom Stateful Iterator Class

Let us build an `ArithmeticProgression` iterator that generates a linear progression of numbers: $a_k = a_0 + k \cdot d$ up to $N$ steps:

In [ ]:
class ArithmeticProgression:
    """Stateful iterator generating an arithmetic sequence: start, start + step, ..."""

    def __init__(self, start: float, step: float, max_terms: int) -> None:
        if max_terms < 0:
            raise ValueError("max_terms must be non-negative.")
        self.start = float(start)
        self.step = float(step)
        self.max_terms = int(max_terms)
        self._current_term = 0
        self._current_value = self.start

    def __iter__(self) -> "ArithmeticProgression":
        return self

    def __next__(self) -> float:
        if self._current_term >= self.max_terms:
            raise StopIteration

        val = self._current_value
        self._current_value += self.step
        self._current_term += 1
        return val


# Test custom iterator
prog = ArithmeticProgression(start=2.0, step=3.0, max_terms=5)
series = list(prog)
print(f"Generated series: {series}")
assert series == [2.0, 5.0, 8.0, 11.0, 14.0]

# Notice that an iterator is single-pass (consumable):
second_pass = list(prog)
print(f"Second pass over consumed iterator: {second_pass}")
assert second_pass == []

## 4. Reusable Iterables vs Consumable Iterators

In data processing pipelines (such as reading training epochs), we need to iterate over the dataset multiple times. If `__iter__()` returns `self` and mutates internal state, the object cannot be reused.

To support multiple passes, separate the **Iterable container** from the **Iterator state machine**:

In [ ]:
class ProgressionIterator:
    """Dedicated state machine for one traversal pass."""

    def __init__(self, start: float, step: float, max_terms: int) -> None:
        self.step = step
        self.max_terms = max_terms
        self.current_term = 0
        self.current_value = start

    def __iter__(self) -> "ProgressionIterator":
        return self

    def __next__(self) -> float:
        if self.current_term >= self.max_terms:
            raise StopIteration
        val = self.current_value
        self.current_value += self.step
        self.current_term += 1
        return val


class ProgressionSeries:
    """Reusable Iterable container; creates a new iterator on every pass."""

    def __init__(self, start: float, step: float, max_terms: int) -> None:
        self.start = start
        self.step = step
        self.max_terms = max_terms

    def __iter__(self) -> ProgressionIterator:
        # Every call returns a fresh, unconsumed iterator
        return ProgressionIterator(self.start, self.step, self.max_terms)


reusable_series = ProgressionSeries(start=1.0, step=0.5, max_terms=4)

pass1 = list(reusable_series)
pass2 = list(reusable_series)

print(f"Pass 1: {pass1}")
print(f"Pass 2: {pass2}")
assert pass1 == pass2 == [1.0, 1.5, 2.0, 2.5]

## 5. Generator Functions vs Custom Iterator Classes

Python's `yield` keyword provides a concise alternative to writing custom iterator classes by having the compiler generate the state machine automatically:

In [ ]:
def progression_generator(start: float, step: float, max_terms: int):
    """Generator function equivalent to ArithmeticProgression."""
    val = start
    for _ in range(max_terms):
        yield val
        val += step


gen = progression_generator(start=2.0, step=3.0, max_terms=5)
assert list(gen) == [2.0, 5.0, 8.0, 11.0, 14.0]
print("Generator matches class-based iterator output perfectly!")

## Summary

- **Iterable:** Object with `__iter__()` returning an iterator.
- **Iterator:** Object with `__next__()` producing items and raising `StopIteration`.
- **Desugaring:** `for` loop = `iter()` + `while True:` + `next()` + `try...except StopIteration`.
- **Multi-pass vs One-shot:** To allow multiple passes, separate the iterable container from its transient iterators.